In [1]:
import pandas as pd

df_clean = pd.read_csv("../data_processed/metro_trips_clean.csv")
your_stations = sorted(df_clean['metro_station'].unique())
print(len(your_stations))
for s in your_stations:
    print(s)

53
ADCB Metro Station
Abu Baker Al Siddique Metro Station
Abu Hail Metro Station
Airport Terminal 1 Metro Station
Airport Terminal 3 Metro Station
Al Fahidi Metro Station
Al Furjan Metro Station
Al Ghubaiba Metro Station
Al Jadaf Metro Station
Al Jafiliya Metro Station
Al Nahda Metro Station
Al Qiyadah Metro Station
Al Qusais Metro Station
Al Ras Metro Station
Al Rigga Metro Station
Baniyas Square Metro Station
BurJuman Metro Station
Burj Khalifa/Dubai Mall Metro Station
Business Bay Metro Station
Creek Metro Station
DAMAC Properties Metro Station
DMCC Metro Station
Danube Metro Station
Deira City Centre Metro Station
Discovery Gardens Metro Station
Dubai Airport Free Zone Metro Station
Dubai Healthcare City Metro Station
Dubai Internet City Metro Station
Dubai Investment Park Metro Station
EXPO Metro Station
Emirates Metro Station
Emirates Towers Metro Station
Energy Metro Station
Etisalat Metro Station
Financial Centre Metro Station
First Abu Dhabi Bank Metro Station
GGICO Metro Stat

In [2]:
with open("../docs/station_list.txt", "w") as f:
    for s in your_stations:
        f.write(s + "\n")
print("Written.")

Written.


In [3]:
line_map = {
    'ADCB Metro Station': 'Red',
    'Abu Baker Al Siddique Metro Station': 'Green',
    'Abu Hail Metro Station': 'Green',
    'Airport Terminal 1 Metro Station': 'Red',
    'Airport Terminal 3 Metro Station': 'Red',
    'Al Fahidi Metro Station': 'Green',
    'Al Furjan Metro Station': 'Red (Route 2020)',
    'Al Ghubaiba Metro Station': 'Green',
    'Al Jadaf Metro Station': 'Green',
    'Al Jafiliya Metro Station': 'Red',
    'Al Nahda Metro Station': 'Green',
    'Al Qiyadah Metro Station': 'Green',
    'Al Qusais Metro Station': 'Green',
    'Al Ras Metro Station': 'Green',
    'Al Rigga Metro Station': 'Red',
    'Baniyas Square Metro Station': 'Green',
    'BurJuman Metro Station': 'Red/Green (Interchange)',
    'Burj Khalifa/Dubai Mall Metro Station': 'Red',
    'Business Bay Metro Station': 'Red',
    'Creek Metro Station': 'Green',
    'DAMAC Properties Metro Station': 'Red',
    'DMCC Metro Station': 'Red',
    'Danube Metro Station': 'Red',
    'Deira City Centre Metro Station': 'Red',
    'Discovery Gardens Metro Station': 'Red (Route 2020)',
    'Dubai Airport Free Zone Metro Station': 'Green',
    'Dubai Healthcare City Metro Station': 'Green',
    'Dubai Internet City Metro Station': 'Red',
    'Dubai Investment Park Metro Station': 'Red (Route 2020)',
    'EXPO Metro Station': 'Red (Route 2020)',
    'Emirates Metro Station': 'Red',
    'Emirates Towers Metro Station': 'Red',
    'Energy Metro Station': 'Red',
    'Etisalat Metro Station': 'Green',
    'Financial Centre Metro Station': 'Red',
    'First Abu Dhabi Bank Metro Station': 'Red',   # was "First Gulf Bank"
    'GGICO Metro Station': 'Red',
    'Gold Souq Metro Station': 'Green',            # was Palm Deira
    'Ibn Battuta Metro Station': 'Red',
    'Jabal Ali Metro Station': 'Red',               # was Nakheel Harbour & Tower
    'Jumeirah Golf Estates Metro Station': 'Red (Route 2020)',
    'Mall of the Emirates Metro Station': 'Red',
    'Mashreq Metro Station': 'Red',                 # was Sharaf DG (old)
    'Nakheel Metro Station': 'Red',
    'Noor Bank Metro Station': 'Red',
    'Oud Metha Metro Station': 'Green',
    'Rashidiya Metro Station': 'Red',
    'Salah Al Din Metro Station': 'Green',
    'Stadium Metro Station': 'Green',
    'The Gardens Metro Station': 'Red (Route 2020)',
    'UAE Exchange Metro Station': 'Red',
    'Union Metro Station': 'Red/Green (Interchange)',
    'World Trade Centre Metro Station': 'Red',
}

df_clean['line'] = df_clean['metro_station'].map(line_map)
print(df_clean['line'].value_counts(dropna=False))
print("\nUnmapped stations:", df_clean[df_clean['line'].isna()]['metro_station'].unique())

line
Red                        2567
Green                      1728
Red (Route 2020)            402
Red/Green (Interchange)     192
Name: count, dtype: int64

Unmapped stations: <StringArray>
[]
Length: 0, dtype: str


In [4]:
df_clean.to_csv("../data_processed/metro_trips_clean.csv", index=False)
print("Saved with line column:", df_clean.shape)

Saved with line column: (4889, 8)


In [5]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

geolocator = Nominatim(user_agent="p18389_dubai_metro_dissertation")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.2)

coords = {}
for station in your_stations:
    query = f"{station}, Dubai, UAE"
    location = geocode(query)
    if location:
        coords[station] = (location.latitude, location.longitude)
        print(f"OK: {station} -> {location.latitude:.5f}, {location.longitude:.5f}")
    else:
        coords[station] = (None, None)
        print(f"MISSING: {station}")

OK: ADCB Metro Station -> 25.24321, 55.29832
MISSING: Abu Baker Al Siddique Metro Station
MISSING: Abu Hail Metro Station
MISSING: Airport Terminal 1 Metro Station
MISSING: Airport Terminal 3 Metro Station
MISSING: Al Fahidi Metro Station
OK: Al Furjan Metro Station -> 25.03517, 55.14550
MISSING: Al Ghubaiba Metro Station
MISSING: Al Jadaf Metro Station
MISSING: Al Jafiliya Metro Station
OK: Al Nahda Metro Station -> 25.27271, 55.36939
MISSING: Al Qiyadah Metro Station
MISSING: Al Qusais Metro Station
MISSING: Al Ras Metro Station
OK: Al Rigga Metro Station -> 25.26363, 55.32375
MISSING: Baniyas Square Metro Station
MISSING: BurJuman Metro Station
OK: Burj Khalifa/Dubai Mall Metro Station -> 25.20144, 55.26955
MISSING: Business Bay Metro Station
MISSING: Creek Metro Station
MISSING: DAMAC Properties Metro Station
MISSING: DMCC Metro Station
MISSING: Danube Metro Station
MISSING: Deira City Centre Metro Station
MISSING: Discovery Gardens Metro Station
MISSING: Dubai Airport Free Zone Me

In [6]:
import requests

overpass_url = "https://overpass-api.de/api/interpreter"
# Bounding box roughly covering Dubai: south, west, north, east
overpass_query = """
[out:json][timeout:60];
(
  node["railway"="station"](24.9,54.9,25.35,55.55);
  node["railway"="stop"](24.9,54.9,25.35,55.55);
  node["public_transport"="station"]["railway"](24.9,54.9,25.35,55.55);
);
out body;
"""

headers = {'User-Agent': 'p18389-dubai-metro-dissertation/1.0'}
response = requests.post(overpass_url, data={'data': overpass_query}, headers=headers)
print(response.status_code)

data = response.json()
print("Stations found:", len(data['elements']))
for el in data['elements'][:15]:
    print(el.get('tags', {}).get('name'), '->', el['lat'], el['lon'])

200
Stations found: 204
Financial Center -> 25.2109196 55.2754808
ماكس -> 25.2333634 55.2920421
دبي مول / برج خليفة -> 25.2014367 55.2695451
دهانات ناشونال -> 25.0579909 55.1271958
أبراج الإمارات -> 25.217231 55.279881
المطار - مبنى رقم 1 -> 25.2484962 55.3523482
سوق التأمين -> 25.1147866 55.1908843
جارمن -> 25.1556519 55.2284999
مدينة دبي للإنترنت -> 25.1022098 55.1738689
شوبا العقارية -> 25.079961 55.1475864
الفردان للصرافة -> 25.0889623 55.1581441
إكويتي -> 25.1266938 55.2079227
مول الإمارات -> 25.1212346 55.2004313
الرقة -> 25.2633609 55.323887
دانوب -> 25.0012917 55.0956664


In [8]:
overpass_query = """
[out:json][timeout:60];
(
  node["railway"="station"]["station"="subway"](24.9,54.9,25.35,55.55);
);
out body;
"""

response = requests.post(overpass_url, data={'data': overpass_query}, headers=headers)
data = response.json()
print("Stations found:", len(data['elements']))

osm_stations = {}
for el in data['elements']:
    tags = el.get('tags', {})
    name_en = tags.get('name:en') or tags.get('name')
    osm_stations[name_en] = (el['lat'], el['lon'])
    print(name_en, '->', el['lat'], el['lon'])

Stations found: 64
Burj Khalifa / Dubai Mall -> 25.2014367 55.2695451
Emirates Towers -> 25.217231 55.279881
Airport Terminal 3 -> 25.245027 55.3594972
Business Bay -> 25.1913122 55.2603981
Al Furjan -> 25.0304736 55.1522307
Discovery Gardens -> 25.0352571 55.1454134
e& -> 25.2547338 55.4012781
Al Nahda -> 25.2732083 55.3694757
Stadium -> 25.27791 55.361466
Abu Hail -> 25.2754161 55.3464499
Al Qiyadah -> 25.2775853 55.3526309
Abu Baker Al Siddique -> 25.2708929 55.3327545
Union -> 25.2661816 55.3137508
Salah Al Din -> 25.2703152 55.3208284
Baniyas Square -> 25.2691743 55.3076094
Gold Souq -> 25.2761408 55.3016752
Al Ras -> 25.2687742 55.2936231
Al Ghubaiba -> 25.2650351 55.2889404
Sharaf DG -> 25.2582139 55.2974781
BurJuman -> 25.25499 55.303941
Oud Metha -> 25.2439301 55.3158939
Dubai Healthcare City -> 25.2309528 55.322619
Al Jadaf -> 25.2249103 55.3337479
Creek -> 25.2188806 55.3390384
Al Qusais -> 25.2627427 55.3872735
Dubai Airport Free Zone -> 25.2698284 55.3751502
Danube -> 25.0

In [9]:
with open("../docs/osm_stations.txt", "w", encoding="utf-8") as f:
    for name, (lat, lon) in osm_stations.items():
        f.write(f"{name}\t{lat}\t{lon}\n")
print("Written", len(osm_stations), "stations")

Written 64 stations


In [10]:
osm_stations['Al Fahidi Metro Station'] = osm_stations.pop('Sharaf DG')
osm_stations['DAMAC Properties Metro Station'] = osm_stations.pop('Sobha Realty')
osm_stations['EXPO Metro Station'] = osm_stations.pop('Expo City Dubai')
osm_stations['Etisalat Metro Station'] = osm_stations.pop('e&')
osm_stations['Mashreq Metro Station'] = osm_stations.pop('InsuranceMarket')
osm_stations['Rashidiya Metro Station'] = osm_stations.pop('Centrepoint')
osm_stations['UAE Exchange Metro Station'] = osm_stations.pop('Life Pharmacy')
osm_stations['Al Jafiliya Metro Station'] = osm_stations.pop('max')

# Add " Metro Station" suffix to remaining direct-name matches so they align with your dataset
suffix_matches = {}
for name, coord in osm_stations.items():
    if not name.endswith('Metro Station'):
        suffix_matches[name + ' Metro Station'] = coord
    else:
        suffix_matches[name] = coord

matched = {s: suffix_matches[s] for s in your_stations if s in suffix_matches}
unresolved = [s for s in your_stations if s not in suffix_matches]

print(f"Matched: {len(matched)} / {len(your_stations)}")
print("Still unresolved:", unresolved)

Matched: 47 / 53
Still unresolved: ['Burj Khalifa/Dubai Mall Metro Station', 'First Abu Dhabi Bank Metro Station', 'GGICO Metro Station', 'Jabal Ali Metro Station', 'Nakheel Metro Station', 'Noor Bank Metro Station']


In [11]:
suffix_matches['Burj Khalifa/Dubai Mall Metro Station'] = suffix_matches.pop('Burj Khalifa / Dubai Mall Metro Station')
matched = {s: suffix_matches[s] for s in your_stations if s in suffix_matches}
unresolved = [s for s in your_stations if s not in suffix_matches]
print(f"Matched: {len(matched)} / {len(your_stations)}")
print("Still unresolved:", unresolved)

Matched: 48 / 53
Still unresolved: ['First Abu Dhabi Bank Metro Station', 'GGICO Metro Station', 'Jabal Ali Metro Station', 'Nakheel Metro Station', 'Noor Bank Metro Station']


In [12]:
final_rename = {
    'Equiti Metro Station': 'First Abu Dhabi Bank Metro Station',
    'Al Garhoud Metro Station': 'GGICO Metro Station',
    'National Paints Metro Station': 'Jabal Ali Metro Station',
    'Al Fardan Exchange Metro Station': 'Nakheel Metro Station',
    'Garmin Metro Station': 'Noor Bank Metro Station',
}

for osm_name, your_name in final_rename.items():
    if osm_name in suffix_matches:
        suffix_matches[your_name] = suffix_matches[osm_name]

matched = {s: suffix_matches[s] for s in your_stations if s in suffix_matches}
unresolved = [s for s in your_stations if s not in suffix_matches]
print(f"Matched: {len(matched)} / {len(your_stations)}")
print("Still unresolved:", unresolved)

Matched: 53 / 53
Still unresolved: []


In [13]:
coords_df = pd.DataFrame([
    {'metro_station': s, 'latitude': lat, 'longitude': lon}
    for s, (lat, lon) in matched.items()
])
coords_df = coords_df.sort_values('metro_station').reset_index(drop=True)
print(coords_df.shape)
print(coords_df.head(10))

coords_df.to_csv("../data_processed/station_coordinates.csv", index=False)
print("Saved.")

(53, 3)
                         metro_station   latitude  longitude
0                   ADCB Metro Station  25.244510  55.298183
1  Abu Baker Al Siddique Metro Station  25.270893  55.332755
2               Abu Hail Metro Station  25.275416  55.346450
3     Airport Terminal 1 Metro Station  25.248477  55.352356
4     Airport Terminal 3 Metro Station  25.245027  55.359497
5              Al Fahidi Metro Station  25.258214  55.297478
6              Al Furjan Metro Station  25.030474  55.152231
7            Al Ghubaiba Metro Station  25.265035  55.288940
8               Al Jadaf Metro Station  25.224910  55.333748
9            Al Jafiliya Metro Station  25.233594  55.292184
Saved.


In [14]:
df_clean = df_clean.merge(coords_df, on='metro_station', how='left')
print("Any missing coordinates after merge:", df_clean['latitude'].isna().sum())
df_clean.to_csv("../data_processed/metro_trips_clean.csv", index=False)
print("Final saved:", df_clean.shape)

Any missing coordinates after merge: 0
Final saved: (4889, 10)
